# Minimal RAG Pipeline

This notebook loads documents, chunks them, stores embeddings in Pinecone, retrieves similar chunks for a query, and generates an answer with an LLM.

## Imports

This cell loads the libraries used by the notebook, including the optional readers for PDF and DOCX files.

In [1]:
from __future__ import annotations

import csv
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

try:
    from docx import Document as DocxDocument
except ImportError:
    DocxDocument = None

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

d:\Miniconda\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Environment and Config

This cell loads environment variables and defines the model, chunking, and Pinecone settings used throughout the notebook.

In [ ]:
load_dotenv()

SUPPORTED_EXTENSIONS = {'.txt', '.md', '.pdf', '.docx', '.csv'}
DEFAULT_EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL_NAME', 'all-MiniLM-L6-v2')
DEFAULT_CHAT_MODEL = os.getenv('GROQ_CHAT_MODEL', 'llama-3.3-70b-versatile')
DEFAULT_CHUNK_SIZE = 1000
DEFAULT_CHUNK_OVERLAP = 150

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
PINECONE_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME', 'rag-minimal')
PINECONE_CLOUD = os.getenv('PINECONE_CLOUD', 'aws')
PINECONE_REGION = os.getenv('PINECONE_REGION', 'us-east-1')

## Document Loading and Chunking

This section reads text from each supported file type and splits it into overlapping chunks for embedding.

In [3]:
@dataclass(frozen=True)
class Chunk:
    id: str
    text: str
    source: str
    chunk_index: int


def load_text_from_file(path: Path) -> str:
    suffix = path.suffix.lower()

    if suffix in {'.txt', '.md'}:
        return path.read_text(encoding='utf-8', errors='ignore')

    if suffix == '.pdf':
        if PdfReader is None:
            raise RuntimeError('pypdf is required to read PDF files.')
        reader = PdfReader(str(path))
        pages = []
        for page in reader.pages:
            pages.append(page.extract_text() or '')
        return '\n'.join(pages)

    if suffix == '.docx':
        if DocxDocument is None:
            raise RuntimeError('python-docx is required to read DOCX files.')
        document = DocxDocument(str(path))
        return '\n'.join(paragraph.text for paragraph in document.paragraphs)

    if suffix == '.csv':
        rows = []
        with path.open('r', encoding='utf-8', errors='ignore', newline='') as handle:
            reader = csv.reader(handle)
            for row in reader:
                rows.append(', '.join(cell.strip() for cell in row if cell.strip()))
        return '\n'.join(rows)

    raise ValueError(f'Unsupported file type: {path.suffix}')


def iter_documents(documents_dir: Path) -> Iterable[Path]:
    for path in documents_dir.rglob('*'):
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS:
            yield path


def chunk_text(text: str, chunk_size: int = DEFAULT_CHUNK_SIZE, overlap: int = DEFAULT_CHUNK_OVERLAP) -> list[str]:
    cleaned = ' '.join(text.split())
    if not cleaned:
        return []

    chunks = []
    start = 0
    text_length = len(cleaned)

    while start < text_length:
        end = min(start + chunk_size, text_length)
        chunk = cleaned[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= text_length:
            break
        start = max(0, end - overlap)

    return chunks


def build_chunks(documents_dir: Path) -> list[Chunk]:
    chunks = []
    for document_path in iter_documents(documents_dir):
        text = load_text_from_file(document_path)
        for chunk_index, chunk_text_value in enumerate(chunk_text(text)):
            chunks.append(
                Chunk(
                    id=f'{document_path.stem}-{chunk_index}',
                    text=chunk_text_value,
                    source=str(document_path.relative_to(documents_dir)),
                    chunk_index=chunk_index,
                )
            )
    return chunks

## Pinecone and Embeddings

This section creates the OpenAI and Pinecone clients, makes sure the index exists, and converts text into embeddings.

In [4]:
def get_groq_client() -> Groq:
    if not GROQ_API_KEY:
        raise RuntimeError('GROQ_API_KEY is required.')
    return Groq(api_key=GROQ_API_KEY)


def get_pinecone_index() -> tuple[Pinecone, str]:
    if not PINECONE_API_KEY:
        raise RuntimeError('PINECONE_API_KEY is required.')
    return Pinecone(api_key=PINECONE_API_KEY), PINECONE_INDEX_NAME


def ensure_index(pc: Pinecone, index_name: str, dimension: int) -> None:
    existing_indexes = [index.name for index in pc.list_indexes()]
    if index_name in existing_indexes:
        return

    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric='cosine',
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )


_embedding_model: SentenceTransformer | None = None

def get_embedding_model(model_name: str | None = None) -> SentenceTransformer:
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer(model_name or DEFAULT_EMBEDDING_MODEL)
    return _embedding_model


def embed_texts(texts: list[str], model_name: str | None = None) -> list[list[float]]:
    model = get_embedding_model(model_name)
    embeddings = model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
    return [e.tolist() for e in embeddings]

## Indexing and Retrieval

This section stores chunks in Pinecone, retrieves the nearest matches for a query, and builds the final answer from retrieved context.

In [5]:
def index_documents(documents_dir: Path, namespace: str = 'default') -> int:
    pc, index_name = get_pinecone_index()

    chunks = build_chunks(documents_dir)
    if not chunks:
        return 0

    sample_embedding = embed_texts([chunks[0].text])[0]
    ensure_index(pc, index_name, len(sample_embedding))
    index = pc.Index(index_name)

    batch_size = int(os.getenv('PINECONE_BATCH_SIZE', '32'))
    upserted = 0

    for start in range(0, len(chunks), batch_size):
        batch = chunks[start : start + batch_size]
        embeddings = embed_texts([chunk.text for chunk in batch])
        vectors = []
        for chunk, embedding in zip(batch, embeddings, strict=True):
            vectors.append(
                {
                    'id': f'{chunk.source}:{chunk.chunk_index}',
                    'values': embedding,
                    'metadata': {
                        'text': chunk.text,
                        'source': chunk.source,
                        'chunk_index': chunk.chunk_index,
                    },
                }
            )
        index.upsert(vectors=vectors, namespace=namespace)
        upserted += len(vectors)

    return upserted


def retrieve_context(query: str, top_k: int = 5, namespace: str = 'default') -> list[dict[str, object]]:
    pc, index_name = get_pinecone_index()
    index = pc.Index(index_name)

    query_embedding = embed_texts([query])[0]
    result = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        namespace=namespace,
    )
    matches = result.matches if hasattr(result, 'matches') else result.get('matches', [])

    contexts = []
    for match in matches:
        metadata = match.get('metadata', {}) if isinstance(match, dict) else getattr(match, 'metadata', {})
        score = match.get('score', 0.0) if isinstance(match, dict) else getattr(match, 'score', 0.0)
        contexts.append(
            {
                'text': metadata.get('text', ''),
                'source': metadata.get('source', 'unknown'),
                'chunk_index': metadata.get('chunk_index', -1),
                'score': score,
            }
        )
    return contexts


def answer_query(query: str, top_k: int = 5, namespace: str = 'default') -> str:
    client = get_groq_client()
    contexts = retrieve_context(query=query, top_k=top_k, namespace=namespace)

    context_block = '\n\n'.join(
        f"Source: {item['source']}\nScore: {item['score']:.4f}\nText: {item['text']}" for item in contexts if item['text']
    )

    messages = [
        {
            'role': 'system',
            'content': (
                'You answer questions using only the provided context. '
                'If the context is insufficient, say what is missing instead of guessing.'
            ),
        },
        {
            'role': 'user',
            'content': f"Question: {query}\n\nContext:\n{context_block or 'No relevant context found.'}",
        },
    ]

    response = client.chat.completions.create(model=DEFAULT_CHAT_MODEL, messages=messages)
    return response.choices[0].message.content or ''

## Example Usage

This final cell shows the commands you can uncomment to index documents and ask a question.

In [6]:
# Example usage
# Create a 'docs' folder and place your PDF files inside, then:
documents_dir = Path('./docs')
indexed = index_documents(documents_dir, namespace='default')
print(f'Indexed {indexed} chunks.')

answer = answer_query('What does the paper say about hybrid rag?', top_k=5, namespace='default')
print(answer)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3413.51it/s]


Indexed 101 chunks.


BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}